# **Download Visual Transformers**

In [1]:
!pip install git+https://github.com/openai/CLIP.git

  Cloning https://github.com/openai/CLIP.git to /tmp/pip-req-build-h095s03q
  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git /tmp/pip-req-build-h095s03q
  Resolved https://github.com/openai/CLIP.git to commit a1d071733d7111c9c014f024669f959182114e33
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.4/53.4 kB 1.5 MB/s eta 0:00:00
  Created wheel for clip: filename=clip-1.0-py3-none-any.whl size=1369497 sha256=325653ef40dbe31605529bd37d0b3a9a6340680d90726fec61b9cd2bc4ae77ff
  Stored in directory: /tmp/pip-ephem-wheel-cache-dgh4ynmo/wheels/da/2b/4c/d6691fa9597aac8bb85d2ac13b112deb897d5b50f5ad9a37e4
Successfully built clip


In [2]:
!pip install timm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 13.1 MB/s eta 0:00:00


# **Import Libraries**

In [1]:
import torch
import clip
import timm
from PIL import Image
import os
import torchvision
import tarfile
from torchvision.datasets.utils import download_url
from torch.utils.data import random_split
from torchvision.datasets import ImageFolder
from torchvision.transforms import ToTensor
from torchvision import models
from torchvision import datasets
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as tt
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torchvision.utils import make_grid
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import torchvision.transforms as T
import matplotlib.pyplot as plt
from tqdm import tqdm
import random
%matplotlib inline
matplotlib.rcParams['figure.facecolor'] = '#ffffff'

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
random.seed(0)
np.random.seed(0)
torch.manual_seed(0)

# **Ready VTs**

In [4]:
clip_model, preprocess = clip.load("ViT-B/32", device=device)
clip_model = clip_model.visual

In [5]:
torch_vision_model = torch.hub.load("pytorch/vision", "resnet50", weights="IMAGENET1K_V2")
torch_vision_model = torch.nn.Sequential(*list(torch_vision_model.children())[:-1])

Using cache found in /root/.cache/torch/hub/pytorch_vision_main


In [6]:
timm_inception_model = timm.create_model('resnet50', pretrained=True, num_classes=0)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:72: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


# **Ready Dataset**

In [9]:
!gdown 1Y-hVa9N-AyabzIc-KKqr5TTwH-tdrwin
!unzip bracs_ood_test.zip
!rm bracs_ood_test.zip
!mv ./final_test ./test
!rm -r ./test/s1
!rm -r ./test/s2
!rm -r ./test/s4
!rm -r ./test/s6
!mv ./test/s0 ./test/0
!mv ./test/s3 ./test/3
!mv ./test/s5 ./test/5

Streaming output truncated to the last 5000 lines.
  inflating: final_test/s2/s2_1016_s2_1_100.png  
  inflating: final_test/s2/s2_1020_s2_1_100.png  
  inflating: final_test/s2/s2_1022_s2_1_100.png  
  inflating: final_test/s2/s2_1023_s2_1_100.png  
  inflating: final_test/s2/s2_1028_s2_1_100.png  
  inflating: final_test/s2/s2_1034_s2_1_112.png  
  inflating: final_test/s2/s2_1040_s2_1_13.png  
  inflating: final_test/s2/s2_1048_s2_1_13.png  
  inflating: final_test/s2/s2_1051_s2_1_13.png  
  inflating: final_test/s2/s2_1053_s2_1_13.png  
  inflating: final_test/s2/s2_1054_s2_1_13.png  
  inflating: final_test/s2/s2_1055_s2_1_13.png  
  inflating: final_test/s2/s2_1061_s2_1_13.png  
  inflating: final_test/s2/s2_1062_s2_1_13.png  
  inflating: final_test/s2/s2_1064_s2_1_13.png  
  inflating: final_test/s2/s2_1067_s2_1_13.png  
  inflating: final_test/s2/s2_1071_s2_1_13.png  
  inflating: final_test/s2/s2_107_s2_0_13.png  
  inflating: final_test/s2/s2_1080_s2_1_13.png  
  inflating: 

# **Model Codes**

In [7]:
class ImageClassificationBase(nn.Module):
    def training_step(self, loss_func, images, positive_images, negative_images):
        embeddings = self(images)
        positive_embeddings = self(positive_images)
        negative_embeddings = self(negative_images)
        loss = loss_func(embeddings, positive_embeddings, negative_embeddings)
        return loss

    def validation_step(self, loss_func, images, positive_images, negative_images):
        embeddings = self(images)
        positive_embeddings = self(positive_images)
        negative_embeddings = self(negative_images)
        loss = loss_func(embeddings, positive_embeddings, negative_embeddings)
        return {'val_loss': loss.detach()}

    def validation_epoch_end(self, outputs):
        batch_losses = [x['val_loss'] for x in outputs]
        epoch_loss = torch.stack(batch_losses).mean()
        return {'val_loss': epoch_loss.item()}

    def epoch_end(self, epoch, result):
        print("Epoch [{}], train_loss: {:.4f}, val_loss: {:.4f}".format(
            epoch, result['train_loss'], result['val_loss']))

In [8]:
def get_default_device():
    if torch.cuda.is_available():
        return torch.device('cuda')
    else:
        return torch.device('cpu')

def to_device(data, device):
    if isinstance(data, (str)):
        return data
    if isinstance(data, (list,tuple)):
        return [to_device(x, device) for x in data]
    return data.to(device, non_blocking=True)

class DeviceDataLoader():
    def __init__(self, dl, device):
        self.dl = dl
        self.device = device

    def __iter__(self):
        for b in self.dl:
            yield to_device(b, self.device)

    def __len__(self):
        return len(self.dl)

In [9]:
device = get_default_device()
device

device(type='cuda')

In [10]:
class Cifar10CnnModel(ImageClassificationBase):
    def __init__(self, vt):
        super().__init__()
        if vt == "clip":
          self.vt = clip_model
          self.embedding = nn.Sequential(
              nn.Linear(512, 256),
              nn.ReLU()
          )
        if vt == "torch vision":
          self.vt = torch_vision_model
          self.embedding = nn.Sequential(
              nn.Flatten(),
              nn.Linear(2048, 256),
              nn.ReLU()
          )
        if vt == "timm inception":
          self.vt = timm_inception_model
          self.embedding = nn.Sequential(
              nn.Linear(2048, 256),
              nn.ReLU()
          )

    def forward(self, xb):
        xb = self.vt(xb)
        embedding = self.embedding(xb)
        return embedding

# **Get Embedding**

In [14]:
!gdown -O ./checkpoint.pth 1-E3ktYvBbkzfvwozDH4MUFtR5W-Juiv0

Downloading...
From: https://drive.google.com/uc?id=1-E3ktYvBbkzfvwozDH4MUFtR5W-Juiv0
To: /content/checkpoint.pth
100% 289M/289M [00:03<00:00, 84.8MB/s]


In [11]:
checkpoint = torch.load("./checkpoint.pth")
model = Cifar10CnnModel("timm inception")
model.load_state_dict(checkpoint['model'])

<All keys matched successfully>

In [12]:
class ReturnIndexDataset(datasets.ImageFolder):
    def __getitem__(self, idx):
        path, target = self.samples[idx]
        image = self.loader(path)
        filename = os.path.basename(path)

        if self.transform is not None:
          image = self.transform(image)

        if self.target_transform is not None:
          target = self.target_transform(target)

        return image, target, filename

In [13]:
test_dataset = ReturnIndexDataset('./test', transform=ToTensor())

In [14]:
embeddings = list()
labels = list()
image_paths = list()

for image, target, filename in tqdm(test_dataset):
  image = image.reshape(1, image.shape[0], image.shape[1], image.shape[2])
  output = model(image)
  output = np.asarray(output.squeeze().detach().cpu())
  embeddings.append(output)
  labels.append(target)
  image_paths.append(filename)

100%|██████████| 3000/3000 [09:22<00:00,  5.33it/s]


In [15]:
embeddings = np.asarray(embeddings)
print(embeddings.shape)
labels = np.asarray(labels)
print(labels.shape)
print(len(image_paths))

(3000, 256)
(3000,)
3000


In [16]:
np.save("./embeddings.npy", embeddings)
np.save("./labels.npy", labels)

import json
image_paths = {"test_file_paths": image_paths}
with open("./file_paths.json", 'w') as f:
  json.dump(image_paths, f)